# 🧬 EGFR Conditional Binder — ESMFold Structure Validation
## Anthropic × Adaptyv Protein Design Competition 2026 — Challenge 1

**Purpose:** Predict 3D structures of our 14 candidate sequences using the ESMFold API.  
**Filter criteria:** pLDDT > 70 overall, pLDDT > 75 for binding helix (first 18 aa)  
**Output:** Filtered sequences for ColabFold multimer validation

---
### Design Summary
- **Target:** EGFR Domain III (conserved human+mouse epitope)
- **Scaffold:** 3-helix bundle miniprotein, 69-79 aa
- **pH Switch:** His residues at interface form salt bridges with EGFR D355/E352 at pH 6.5
- **Cross-species:** Targets conserved Domain III groove (D355, E352, E384 — identical in human/mouse)


In [ ]:
# Install required packages
!pip install requests biopython matplotlib numpy -q

In [ ]:
import requests
import json
import time
import re
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional

print('✅ Libraries loaded')

In [ ]:
# ─── ALL 14 PASSING CANDIDATE SEQUENCES ─────────────────────────────────────
# Generated by rational 3-helix bundle design with histidine-switch mechanism
# pH 6.5: His protonated → salt bridge with EGFR D355/E352 → HIGH affinity
# pH 7.4: His neutral → no contact → LOW affinity

CANDIDATES = [
    # SET A: Standard His at H1 positions 4+8
    {'name': 'EGFR-pH-HB-A01', 'group': 'A', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'FYNAHHLKALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-A02', 'group': 'A', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'WYQAHHLKALLAEELANLKRLEEAQKRLEEQLKGSGSLAAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-A03', 'group': 'A', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'FYNHHALKALLQEELAQMKRLEEAQKRLEQALKGSGSLAAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-A04', 'group': 'A', 'his_count': 3, 'ph_score': 7.77,
     'seq': 'FYNAHHLRALLQQELAQLKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-A05', 'group': 'A', 'his_count': 3, 'ph_score': 7.78,
     'seq': 'YWNAHHLKALLQQELAQMKRLEEAQKRLEQALKGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    # SET B: His at H1 positions 4+8 + buried His in H3 (cooperative switch)
    {'name': 'EGFR-pH-HB-B01', 'group': 'B', 'his_count': 4, 'ph_score': 8.60,
     'seq': 'FYNAHHLKALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEHHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-B02', 'group': 'B', 'his_count': 4, 'ph_score': 8.60,
     'seq': 'WYQAHHLKALLAEELANLKRLEEAQKRLEEQLKGSGSLAAALRRLEQALREQKLGSLEHHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-B03', 'group': 'B', 'his_count': 4, 'ph_score': 8.60,
     'seq': 'FYNAHHLKALLQQELAELKRLEQAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQHLK'},
    
    # SET C: Shifted His positions (5+9 register)
    {'name': 'EGFR-pH-HB-C01', 'group': 'C', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'FYNRAHHLKALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-C03', 'group': 'C', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'FYNKAHHLALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    # SET D: Wide His spacing (positions 4+11)
    {'name': 'EGFR-pH-HB-D01', 'group': 'D', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'FYNKHLAKLHLLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    {'name': 'EGFR-pH-HB-D02', 'group': 'D', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'WYNKHLARLHLLQQELAELKRLEEAQKRLEQALKGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    # SET E: Extended H1 (79 aa)
    {'name': 'EGFR-pH-HB-E01', 'group': 'E', 'his_count': 3, 'ph_score': 8.28,
     'seq': 'FYNAHHLKALLQQELAELKRQLGSGSLEAALRRLEQALREQKLGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
    
    # SET G: Triple His interface (highest pH switch score)
    {'name': 'EGFR-pH-HB-G01', 'group': 'G', 'his_count': 4, 'ph_score': 11.60,
     'seq': 'FYNAHHLKAHLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK'},
]

print(f'✅ Loaded {len(CANDIDATES)} candidate sequences')
print(f'Length range: {min(len(c["seq"]) for c in CANDIDATES)}–{max(len(c["seq"]) for c in CANDIDATES)} aa')
print(f'pH switch score range: {min(c["ph_score"] for c in CANDIDATES):.2f}–{max(c["ph_score"] for c in CANDIDATES):.2f}')

In [ ]:
# ─── ESMFold API STRUCTURE PREDICTION ────────────────────────────────────────
# ESMFold is free, no auth required, handles sequences up to ~400 aa
# Returns PDB format with per-residue pLDDT in B-factor column

ESMATLAS_URL = 'https://api.esmatlas.com/foldSequence/v1/pdb/'

def fold_with_esmfold(sequence: str, name: str, max_retries: int = 3) -> Optional[str]:
    """Fold a sequence using ESMFold API. Returns PDB string or None."""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                ESMATLAS_URL,
                headers={'Content-Type': 'application/x-www-form-urlencoded'},
                data=sequence,
                timeout=120
            )
            if response.status_code == 200:
                return response.text
            else:
                print(f'  ⚠️ {name}: HTTP {response.status_code}, retry {attempt+1}/{max_retries}')
                time.sleep(5)
        except Exception as e:
            print(f'  ❌ {name}: Error {e}, retry {attempt+1}/{max_retries}')
            time.sleep(10)
    return None


def extract_plddt_from_pdb(pdb_string: str) -> List[float]:
    """Extract per-residue pLDDT scores from PDB B-factor column."""
    plddt_scores = []
    seen_residues = set()
    
    for line in pdb_string.split('\n'):
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':  # Alpha carbon only
            chain = line[21]
            res_num = int(line[22:26].strip())
            res_key = (chain, res_num)
            if res_key not in seen_residues:
                b_factor = float(line[60:66].strip())
                plddt_scores.append(b_factor)
                seen_residues.add(res_key)
    
    return plddt_scores


def assess_helix_plddt(plddt_scores: List[float], h1_end: int = 18) -> Dict:
    """Assess pLDDT for different regions of the protein."""
    if not plddt_scores:
        return {}
    
    return {
        'overall_mean': np.mean(plddt_scores),
        'h1_mean': np.mean(plddt_scores[:h1_end]),  # Binding helix
        'h2_mean': np.mean(plddt_scores[h1_end+4:h1_end+24]) if len(plddt_scores) > h1_end+24 else None,  # Core helix
        'min_plddt': min(plddt_scores),
        'fraction_above70': sum(1 for s in plddt_scores if s > 70) / len(plddt_scores),
        'fraction_above80': sum(1 for s in plddt_scores if s > 80) / len(plddt_scores),
    }

print('✅ ESMFold functions defined')

In [ ]:
# ─── RUN ESMFOLD ON ALL 14 CANDIDATES ────────────────────────────────────────
# This takes ~2-3 min for 14 sequences on free Kaggle GPU

results = []
pdb_structures = {}

for i, candidate in enumerate(CANDIDATES):
    name = candidate['name']
    seq = candidate['seq']
    print(f'[{i+1:02d}/{len(CANDIDATES)}] Folding {name} ({len(seq)} aa)...', end=' ')
    
    pdb = fold_with_esmfold(seq, name)
    
    if pdb is None:
        print('❌ FAILED')
        results.append({**candidate, 'esm_status': 'failed', 'plddt_overall': 0})
        continue
    
    pdb_structures[name] = pdb
    plddt_scores = extract_plddt_from_pdb(pdb)
    metrics = assess_helix_plddt(plddt_scores)
    
    overall = metrics.get('overall_mean', 0)
    h1 = metrics.get('h1_mean', 0)
    
    status_icon = '✅' if overall > 70 and h1 > 75 else '⚠️ '
    print(f'{status_icon} pLDDT overall={overall:.1f} | H1={h1:.1f}')
    
    results.append({
        **candidate,
        'esm_status': 'success',
        'plddt_overall': overall,
        'plddt_h1': h1,
        'plddt_min': metrics.get('min_plddt', 0),
        'plddt_frac_above70': metrics.get('fraction_above70', 0),
        'esm_passes': overall > 70 and h1 > 75
    })
    
    # Save PDB structure
    with open(f'{name}.pdb', 'w') as f:
        f.write(pdb)
    
    time.sleep(2)  # Rate limiting

print(f'\n✅ Folding complete: {sum(1 for r in results if r.get("esm_passes"))}/{len(results)} passed')

In [ ]:
# ─── VISUALIZATION: pLDDT SCORES PER CANDIDATE ───────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Overall vs H1 pLDDT
successful = [r for r in results if r.get('esm_status') == 'success']
names = [r['name'].replace('EGFR-pH-HB-', '') for r in successful]
overall_plddt = [r['plddt_overall'] for r in successful]
h1_plddt = [r.get('plddt_h1', 0) for r in successful]
colors = ['#2ecc71' if r.get('esm_passes') else '#e74c3c' for r in successful]

x = np.arange(len(names))
width = 0.35
axes[0].bar(x - width/2, overall_plddt, width, label='Overall pLDDT', color=colors, alpha=0.8)
axes[0].bar(x + width/2, h1_plddt, width, label='H1 Binding Helix pLDDT', 
            color=colors, alpha=0.5, hatch='//')
axes[0].axhline(70, color='orange', linestyle='--', label='Threshold (70)')
axes[0].axhline(75, color='red', linestyle='--', label='H1 Threshold (75)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].set_ylabel('pLDDT Score')
axes[0].set_title('ESMFold Confidence Scores — EGFR Conditional Binder Candidates')
axes[0].legend()
axes[0].set_ylim(0, 100)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: pH Switch Score (pre-computed analytical)
ph_scores = [r['ph_score'] for r in successful]
bars = axes[1].bar(names, ph_scores, color=['#3498db' if s > 8 else '#95a5a6' for s in ph_scores])
axes[1].axhline(3.0, color='red', linestyle='--', label='Min threshold (3.0)')
axes[1].axhline(8.0, color='green', linestyle='--', label='Target (8.0)')
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, rotation=45, ha='right')
axes[1].set_ylabel('pH Switch Score')
axes[1].set_title('pH Conditionality Score (Analytical — Higher = Stronger Tumor-Selective Binding)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('esm_validation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved as esm_validation_results.png')

In [ ]:
# ─── FILTER AND RANK PASSING DESIGNS ─────────────────────────────────────────

passing = [r for r in results if r.get('esm_passes', False)]
passing_sorted = sorted(passing, key=lambda x: -(x['plddt_overall'] * 0.4 + x['ph_score'] * 0.6))

print('=== DESIGNS PASSING ESMFOLD FILTER ===')
print(f'{'Name':<22} {'Len':>4} {'pLDDT':>6} {'H1pLDDT':>8} {'pHscore':>8} {'Group':>6}')
print('-' * 60)

for r in passing_sorted:
    print(f"{r['name']:<22} {len(r['seq']):>4} {r['plddt_overall']:>6.1f} "
          f"{r.get('plddt_h1',0):>8.1f} {r['ph_score']:>8.2f} {r['group']:>6}")

print(f'\n✅ {len(passing)}/{len(results)} designs pass for ColabFold multimer step')
print('\n🔬 Next: Run egfr_colabfold_multimer.ipynb with these sequences')

# Save passing sequences for next step
with open('esm_passing_sequences.json', 'w') as f:
    json.dump(passing_sorted, f, indent=2)
print('💾 Saved passing sequences to esm_passing_sequences.json')

In [ ]:
# ─── SAVE ALL RESULTS ─────────────────────────────────────────────────────────

with open('esm_all_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Summary table for method description
print('\n=== SUMMARY TABLE (for method description PDF) ===')
print(f"{'Design':<22} {'Length':>6} {'pLDDT':>6} {'pH score':>8} {'His':>4} {'Pass':>5}")
print('-' * 55)
for r in results:
    passed = '✅' if r.get('esm_passes') else '❌'
    print(f"{r['name']:<22} {len(r['seq']):>6} {r.get('plddt_overall',0):>6.1f} "
          f"{r['ph_score']:>8.2f} {r['his_count']:>4} {passed:>5}")